# Day 06: PyTorch, Model File Formats, ONNX & TensorRT
> *Inference Engineering* — Chapter 4.2 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 05 (Kernel Fusion)


## Concept Overview

PyTorch is the dominant training framework and increasingly used for inference. Models are persisted in several formats: .pt/.pth (pickle-based PyTorch), SafeTensors (safe, fast tensor-only format), ONNX (cross-framework IR), and TensorRT engines (compiled for specific GPU). The pipeline from research to production typically goes: PyTorch → (optionally) ONNX → TensorRT, with each step trading flexibility for performance. SafeTensors has become the standard for model distribution because it avoids pickle's arbitrary code execution vulnerability.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os, time, struct

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__}')


## 1. PyTorch Model Serialization Formats

Three main formats, each with different tradeoffs on safety, speed, and compatibility:
- **pickle (.pt):** Universal but unsafe (arbitrary code execution on load)
- **SafeTensors:** Fast, safe, header-first format used by HuggingFace
- **ONNX:** Cross-framework IR, required for TensorRT and many deployment runtimes


In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, d=128, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, d*4), nn.GELU(), nn.Linear(d*4, d))
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.ln1(x + a)
        return self.ln2(x + self.ff(x))

model = TinyTransformerBlock()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Save as .pt (state dict)
torch.save(model.state_dict(), '/tmp/model.pt')
size_pt = os.path.getsize('/tmp/model.pt')
print(f'.pt size: {size_pt/1024:.1f} KB')

# Save full model (includes architecture)
torch.save(model, '/tmp/model_full.pt')
size_full = os.path.getsize('/tmp/model_full.pt')
print(f'.pt (full model) size: {size_full/1024:.1f} KB')

# Reload and verify
loaded = TinyTransformerBlock()
loaded.load_state_dict(torch.load('/tmp/model.pt', weights_only=True))
x_test = torch.randn(1, 8, 128)
with torch.no_grad():
    out1 = model(x_test)
    out2 = loaded(x_test)
print(f'State dict reload correct: {torch.allclose(out1, out2, atol=1e-6)}')


## 2. SafeTensors Format

SafeTensors stores a JSON header (tensor shapes/dtypes/offsets) followed by raw bytes. No pickle, no code execution risk. Loading is O(1) for metadata — only tensors you actually read are deserialized.


In [ ]:
try:
    from safetensors.torch import save_file, load_file
    state = model.state_dict()
    save_file(state, '/tmp/model.safetensors')
    size_st = os.path.getsize('/tmp/model.safetensors')
    print(f'SafeTensors size: {size_st/1024:.1f} KB')
    print(f'Size ratio vs .pt: {size_st/size_pt:.2f}x')
    loaded_st = load_file('/tmp/model.safetensors')
    print(f'Keys match: {list(loaded_st.keys()) == list(state.keys())}')
except ImportError:
    print('Installing safetensors...')
    os.system('pip install -q safetensors')
    print('Run again after install')
    # Manually show SafeTensors header structure
    print('\nSafeTensors format:')
    print('  [8 bytes: header_size uint64]')
    print('  [header_size bytes: JSON metadata]')
    print('  [raw tensor bytes, tightly packed]')
    print('  - No pickle, no code execution')
    print('  - Lazy loading: seek to offset, read only needed tensors')


## 3. ONNX Export

ONNX (Open Neural Network Exchange) is a graph IR: nodes are operators, edges are tensors. PyTorch traces the model with example inputs to produce the ONNX graph. Dynamic axes allow variable batch/sequence sizes.


In [ ]:
x_sample = torch.randn(1, 8, 128)
try:
    torch.onnx.export(
        model, x_sample, '/tmp/model.onnx',
        input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0:'batch',1:'seq'}, 'output': {0:'batch',1:'seq'}},
        opset_version=17
    )
    size_onnx = os.path.getsize('/tmp/model.onnx')
    print(f'ONNX size: {size_onnx/1024:.1f} KB')
    print(f'Export successful. Dynamic axes: batch, seq')
except Exception as e:
    print(f'ONNX export: {e}')

# Show ONNX graph loading
try:
    import onnx
    m = onnx.load('/tmp/model.onnx')
    ops = list(set(n.op_type for n in m.graph.node))
    print(f'ONNX graph ops ({len(m.graph.node)} nodes): {sorted(ops)}')
except ImportError:
    print('pip install onnx for graph inspection')

print('\nFormat comparison:')
for fmt, size in [('.pt state_dict', size_pt), ('SafeTensors', size_pt)]:
    print(f'  {fmt}: {size/1024:.1f} KB')


## 4. TensorRT Compilation Pipeline

TensorRT takes an ONNX model, profiles it on the target GPU, and compiles an optimized engine. The engine is GPU-specific and must be recompiled for each GPU architecture. Key optimizations: kernel selection, fusion, precision calibration (FP8/INT8).


In [ ]:
# Conceptual TensorRT pipeline (show without requiring TRT install)
print('TensorRT compilation pipeline:')
print()
steps = [
    ('1. Export ONNX', 'torch.onnx.export(model, sample, model.onnx)'),
    ('2. Parse ONNX',  'parser.parse(open(model.onnx).read())'),
    ('3. Set precision','config.set_flag(trt.BuilderFlag.FP16)'),
    ('4. Build engine','engine = builder.build_engine(network, config)'),
    ('5. Serialize',   'open(model.engine, wb).write(engine.serialize())'),
    ('6. Deploy',      'runtime.deserialize_engine(open(model.engine).read())'),
]
for step, code in steps:
    print(f'  {step:<25} → {code}')

print()
print('Typical speedups vs eager PyTorch (on A100):')
for scenario, speedup in [('FP32 → FP16', '1.8-2.5x'), ('+ kernel fusion', '2.5-4x'), ('+ INT8 calibration', '4-8x')]:
    print(f'  {scenario:<30} {speedup}')


## Experiments: Try These

1. **Format benchmark**: Time the load of a SafeTensors file vs a pickled .pt file for a 1GB model. Measure cold vs warm reads.
2. **ONNX graph inspection**: Export a larger model to ONNX and visualize with Netron (netron.app). Count fused vs unfused ops.
3. **torch.export**: Try `torch.export.export()` (new AOT export API) and compare the graph to ONNX. What operators look different?


## Key Takeaways

- `.pt` files use Python pickle — fast but unsafe; never load untrusted `.pt` files.
- SafeTensors is the standard for model distribution: safe, fast (lazy load), header-first format.
- ONNX is the interoperability IR: PyTorch traces the model with dummy inputs to produce a cross-framework graph.
- TensorRT compiles ONNX to GPU-specific engines with kernel fusion and precision optimization — required for production-grade latency.

## References
- *Inference Engineering* Ch 4.2 — Philip Kiely, Baseten Books 2026
- HuggingFace SafeTensors specification
- NVIDIA TensorRT Developer Guide
